# Case study: missing product prices, imputation and univariate outliers

*Session 4, block 2 · Author: course team · Licence: CC-BY-4.0*

**Questions of the practice block**

1. Is a missing product price related to the number of reviews? What does that say about the missing-data
   mechanism (MCAR, MAR, MNAR)?
2. How well do simple, KNN and iterative imputation recover prices that we hide on purpose?
3. Which prices and review lengths are outliers by the IQR rule, the z-score and the MAD rule?

In [ ]:
from pathlib import Path

# find the repository root (the folder that contains case-study/), wherever the notebook runs
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir())
DATA = ROOT / "case-study" / "data"
assert (DATA / "train.parquet").exists(), "run case-study/prepare_data.py first"

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

products = pd.read_parquet(DATA / "products.parquet")
print(f"{len(products):,} products, price missing for {products['price'].isna().mean():.1%}")

## 1. Turn missingness into a variable

A **missingness indicator** is 1 where the value is missing. Comparing it across other variables tells us
whether the rows with and without a price differ. If the missing share varies with an observed variable,
**MCAR** (missing completely at random) is implausible.

In [ ]:
products["price_missing"] = products["price"].isna()
products["n_reviews"] = products["train_n_reviews"].fillna(0)        # NaN = no training review
popularity = pd.cut(products["n_reviews"], [-1, 0, 1, 5, 20, 100, np.inf],
                    labels=["0", "1", "2-5", "6-20", "21-100", ">100"])
by_pop = products.groupby(popularity, observed=True)["price_missing"].agg(share_missing="mean", n="size")
by_pop.round(3)

In [ ]:
ax = by_pop["share_missing"].plot.bar(color="#2a78d6", rot=0, figsize=(6, 3))
ax.set(xlabel="training reviews per product", ylabel="share with missing price", ylim=(0, 1))
ax.axhline(products["price_missing"].mean(), color="#52514e", ls="--", lw=1)
ax.spines[["top", "right"]].set_visible(False)
plt.show()

**A test of independence.** The chi-square test asks whether the missing share is the same in all
popularity groups (null hypothesis: missingness is independent of popularity). With 60,000 products, even
small differences are significant, so look at the size of the differences as well (applied recap from the
statistics module).

In [ ]:
from scipy.stats import chi2_contingency

table = pd.crosstab(popularity, products["price_missing"])
chi2, p_value, dof, _ = chi2_contingency(table)
print(table)
print(f"chi2 = {chi2:.0f}, dof = {dof}, p = {p_value:.1e}")

In [ ]:
# other observed variables: product page completeness and average rating
products["has_features"] = products["features"].ne("")
products["has_description"] = products["description"].ne("")
print(products.groupby("has_features")["price_missing"].mean().round(3))
print(products.groupby("price_missing")["train_avg_rating"].mean().round(2))

**Can the observed columns predict missingness?** If a classifier predicts the indicator better than chance
(area under the ROC curve, AUC, above 0.5), the data are not MCAR. The variables it uses are the ones an
imputation model should include. (Logistic regression and AUC are taught in Sessions 6 and 8; here we only
read the number.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

features = pd.DataFrame({
    "log_n_reviews": np.log1p(products["n_reviews"]),
    "avg_rating": products["train_avg_rating"].fillna(products["train_avg_rating"].median()),
    "has_features": products["has_features"].astype(int),
    "has_description": products["has_description"].astype(int),
    "log_title_len": np.log1p(products["title"].str.len()),
})
auc = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
                      features, products["price_missing"], cv=5, scoring="roc_auc")
print(f"AUC for predicting a missing price: {auc.mean():.2f}")

**Interpretation.** Missing prices are much more common for products with few reviews and with an empty
feature list. So the prices are **not MCAR**. They may be **MAR** (explained by observed variables such as
popularity) or **MNAR** (for example, if the price is missing *because* a product was very cheap or very
expensive or no longer sold). The data alone cannot distinguish MAR from MNAR.

**TODO 1.** Write two sentences for the data card: what does the missingness pattern mean for an analysis
of "average product price" that uses only products with a price?

## 2. Compare imputation methods on hidden values

We cannot check an imputation against the truth on the real missing prices. Instead we take the products
*with* a price, hide 30 % of them, impute, and compare with the hidden values. We work with the logarithm of
the price, because prices are strongly right-skewed.

In [ ]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, KNNImputer, SimpleImputer

known = features[products["price"].notna()].assign(log_price=np.log(products.loc[products["price"].notna(), "price"]))
known = known.reset_index(drop=True)
rng = np.random.default_rng(2026)
hide = rng.random(len(known)) < 0.3                    # MCAR hiding, for a fair comparison
X = known.copy()
X.loc[hide, "log_price"] = np.nan
truth = known.loc[hide, "log_price"].to_numpy()

imputers = {
    "mean": SimpleImputer(strategy="mean"),
    "median": SimpleImputer(strategy="median"),
    "KNN (k=10, scaled)": make_pipeline(StandardScaler(), KNNImputer(n_neighbors=10)),
    "iterative": IterativeImputer(random_state=0),
}
rows = []
for name, imputer in imputers.items():
    filled = imputer.fit_transform(X)
    if name.startswith("KNN"):                      # undo the scaling for the price column
        scaler = imputer[0]
        filled = filled * scaler.scale_ + scaler.mean_
    pred = filled[hide, list(X.columns).index("log_price")]
    rows.append({"method": name, "RMSE (log price)": np.sqrt(np.mean((pred - truth) ** 2)),
                 "SD of imputed values": pred.std(), "SD of true values": truth.std()})

In [ ]:
# a domain rule: the median price of the same store (falls back to the overall median)
store = products.loc[products["price"].notna(), "store"].reset_index(drop=True)
store_median = known.loc[~hide].groupby(store[~hide])["log_price"].median()
pred = store[hide].map(store_median).fillna(known.loc[~hide, "log_price"].median()).to_numpy()
rows.append({"method": "store median", "RMSE (log price)": np.sqrt(np.mean((pred - truth) ** 2)),
             "SD of imputed values": pred.std(), "SD of true values": truth.std()})
pd.DataFrame(rows).round(3)

**Interpretation.** The observed variables say little about the price, so the model-based imputers barely
beat the mean, and the store median helps a little. Two lessons:

- Imputation cannot create information that the other variables do not contain.
- Mean and median imputation give every missing row the same value: the spread of the imputed values is 0,
  far below the true spread. Analyses of variability or correlation become biased.

**TODO 2.** Add `add_indicator=True` to the `SimpleImputer` and look at the extra column. Why is the
indicator useful for a model even when the imputed value itself is poor?

In [ ]:
# TODO 2

## 3. Univariate outliers: IQR rule, z-score and MAD

Three rules flag a value as an outlier:

- **IQR rule (Tukey):** below Q1 − 1.5·IQR or above Q3 + 1.5·IQR;
- **z-score:** |x − mean| / SD > 3;
- **robust z-score (MAD):** 0.6745 · |x − median| / MAD > 3.5, with MAD = median(|x − median|).

In [ ]:
def outlier_flags(x: pd.Series) -> pd.DataFrame:
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    z = (x - x.mean()) / x.std()
    mad = (x - x.median()).abs().median()
    robust_z = 0.6745 * (x - x.median()) / mad if mad > 0 else pd.Series(np.nan, index=x.index)
    return pd.DataFrame({"iqr": (x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr),
                         "z": z.abs() > 3,
                         "mad": robust_z.abs() > 3.5})


price = products["price"].dropna()
summary = pd.DataFrame({
    "price": outlier_flags(price).sum(),
    "log price": outlier_flags(np.log(price)).sum(),
})
summary

In [ ]:
# what are the flagged prices on the log scale?
flags = outlier_flags(np.log(price))
products.loc[flags.index[flags["mad"]], ["title", "price"]].sort_values("price").iloc[[0, 1, 2, -3, -2, -1]]

On the raw scale, the IQR rule flags hundreds of ordinary prices above about 72 dollars, because the
distribution is right-skewed. On the log scale, the rules flag far fewer products, and those are the
cheapest single items and the most expensive devices, such as mobility scooters. These are genuine values,
not errors: keep them, and use a log transformation or a robust method in the analysis.

**TODO 3.** Apply `outlier_flags` to the review length (`text.str.len()`) and to `helpful_vote` of
`train_sample.parquet`. Why does the MAD rule fail for `helpful_vote`?

In [ ]:
reviews = pd.read_parquet(DATA / "train_sample.parquet")
# TODO 3